# Simpson's paradox trong dữ liệu Olist thật

Khách ở bang SP có tỉ lệ review xấu thấp hơn khách ở các bang khác. "Khách SP dễ tính hơn"? Notebook
này xây công cụ để kiểm tra một so sánh tỉ lệ có đổi chiều khi phân tầng không, rồi áp lên Olist.

Định nghĩa dùng chung của track: đơn giao xong (`delivered` và có ngày khách nhận), review mới nhất
mỗi đơn, review xấu là `review_score <= 2`, giao trễ là so nguyên timestamp với ngày hẹn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/09-eda-storytelling/exercises/intermediate/02-simpson-paradox -q`

In [ ]:
import numpy as np
import pandas as pd

DAY_BINS = [0, 7, 14, 21, 30, np.inf]
DAY_LABELS = ["0-6", "7-13", "14-20", "21-29", "30+"]

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")

## 1. Bảng phân tích mức đơn

In [ ]:
def build_orders(orders: pd.DataFrame, reviews: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong có review. Cột: order_id, customer_state, late, bad, delivery_days.

    giao xong: order_status == "delivered" và có order_delivered_customer_date
    review: mỗi đơn giữ review có review_answer_timestamp mới nhất; bỏ đơn không có review
    late: order_delivered_customer_date > order_estimated_delivery_date (so nguyên timestamp)
    bad: review_score <= 2
    delivery_days: (order_delivered_customer_date − order_purchase_timestamp) tính bằng ngày (float)
    """
    df = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
    df = (df.merge(latest[["order_id", "review_score"]], on="order_id")
            .merge(customers[["customer_id", "customer_state"]], on="customer_id"))
    return pd.DataFrame({
        "order_id": df.order_id,
        "customer_state": df.customer_state,
        "late": df.order_delivered_customer_date > df.order_estimated_delivery_date,
        "bad": df.review_score <= 2,
        "delivery_days": (df.order_delivered_customer_date - df.order_purchase_timestamp).dt.total_seconds() / 86400,
    }).reset_index(drop=True)


def delivery_bucket(days: pd.Series) -> pd.Series:
    """Nhóm số ngày giao theo DAY_BINS (khoảng [a, b), trái đóng phải mở), nhãn DAY_LABELS, có thứ tự."""
    return pd.cut(days, DAY_BINS, right=False, labels=DAY_LABELS)

In [ ]:
# explore: kiểm tra nhanh bảng đơn
df = build_orders(orders, reviews, customers)
df["region"] = np.where(df.customer_state == "SP", "SP", "ngoài SP")
df["days"] = delivery_bucket(df.delivery_days)
print(len(df), df.late.mean().round(4), df.bad.mean().round(4))
df.groupby("region").bad.agg(["mean", "size"])

## 2. Tỉ lệ theo tầng, chuẩn hoá trực tiếp, và kiểm tra đảo chiều

Công thức chuẩn hoá: tỉ lệ chuẩn hoá của nhóm g là tổng theo tầng s của w_s nhân tỉ lệ của g ở tầng s, với cùng bộ trọng số w cho mọi nhóm.

In [ ]:
def stratified_rates(df: pd.DataFrame, group: str, stratum: str, target: str) -> pd.DataFrame:
    """Tỉ lệ target theo (stratum, group). Trả DataFrame cột [stratum, group, "n", "rate"] (tên cột
    stratum và group giữ nguyên tên gốc), sắp theo stratum rồi group; chỉ gồm cặp có dữ liệu."""
    out = (df.groupby([stratum, group], observed=True)[target]
             .agg(n="size", rate="mean")
             .reset_index())
    out["rate"] = out["rate"].astype(float)
    return out.sort_values([stratum, group]).reset_index(drop=True)


def standardized_rates(df: pd.DataFrame, group: str, stratum: str, target: str,
                       weights: pd.Series | None = None) -> pd.Series:
    """Direct standardization: với mỗi group g, sum_s w_s * rate(g, s).

    weights: Series index là giá trị stratum. None → phân bố stratum của chính df (tỉ trọng số dòng).
    Chỉ dùng các stratum có mặt ở MỌI group (và có trong weights); chuẩn hoá lại trọng số của các
    stratum đó cho tổng bằng 1. Trả Series index là các group.
    """
    table = df.groupby([stratum, group], observed=True)[target].mean().astype(float).unstack(group)
    if weights is None:
        weights = df[stratum].value_counts(normalize=True)
    # reindex thay vì .loc: stratum kiểu bool (late) sẽ bị .loc hiểu thành mask
    keep = [s for s in table.dropna().index if s in weights.index]
    w = weights.reindex(keep)
    w = w / w.sum()
    return table.reindex(keep).mul(w, axis=0).sum()


def simpson_check(df: pd.DataFrame, group: str, stratum: str, target: str, a, b) -> dict:
    """So sánh group a với group b (chỉ giữ hai group này trong df).

    crude_diff           rate(a) − rate(b) trên dữ liệu gộp
    adjusted_diff        standardized(a) − standardized(b), trọng số mặc định của standardized_rates
    n_strata             số stratum có mặt ở cả a và b
    strata_against_crude số stratum trong đó rate(a, s) − rate(b, s) khác 0 và ngược dấu crude_diff
    reversal             True nếu crude_diff và adjusted_diff đều khác 0 và ngược dấu
    """
    sub = df[df[group].isin([a, b])]
    crude = sub.groupby(group)[target].mean().astype(float)
    crude_diff = float(crude[a] - crude[b])
    std = standardized_rates(sub, group, stratum, target)
    adjusted_diff = float(std[a] - std[b])
    table = sub.groupby([stratum, group], observed=True)[target].mean().astype(float).unstack(group).dropna()
    per_stratum = table[a] - table[b]
    against = int(((per_stratum != 0) & (np.sign(per_stratum) == -np.sign(crude_diff))).sum())
    return {
        "crude_diff": crude_diff,
        "adjusted_diff": adjusted_diff,
        "n_strata": int(len(table)),
        "strata_against_crude": against,
        "reversal": bool(crude_diff != 0 and adjusted_diff != 0 and np.sign(crude_diff) != np.sign(adjusted_diff)),
    }

## 3. SP và các bang khác, chia theo số ngày giao

In [ ]:
# explore: bảng theo tầng và tỉ trọng mỗi tầng trong từng nhóm
rates = stratified_rates(df, "region", "days", "bad")
display(rates.pivot(index="days", columns="region", values="rate").round(4))
display(pd.crosstab(df.days, df.region, normalize="columns").round(3))
simpson_check(df, "region", "days", "bad", "SP", "ngoài SP")

In [ ]:
# explore: vẽ tỉ lệ theo tầng (thứ tự tự nhiên của bucket, không sắp theo giá trị)
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))
sns.pointplot(data=df, x="days", y="bad", hue="region", errorbar=("ci", 95), ax=ax)
ax.set_title("Cùng số ngày giao, khách SP không dễ tính hơn")
ax.set_xlabel("Số ngày từ lúc đặt tới lúc nhận")
ax.set_ylabel("Tỉ lệ review 1–2 sao")
plt.show()

## 4. Cùng dữ liệu, chia theo trễ / đúng hẹn

In [ ]:
# explore
display(stratified_rates(df, "region", "late", "bad"))
simpson_check(df, "region", "late", "bad", "SP", "ngoài SP")

## 5. Kết luận (bài mẫu)

**Gộp lại, khách SP có tỉ lệ review xấu thấp hơn 3.7 điểm phần trăm (10.7% so với 14.3%). Ở cùng số
ngày giao, khách SP lại có tỉ lệ cao hơn ở 4/5 nhóm, và chuẩn hoá theo cùng phân bố số ngày giao thì
SP cao hơn 2.9 điểm.** Lý do: 46.9% đơn SP được giao dưới 7 ngày, so với 12.7% ở các bang khác. Con
số gộp đo "SP được giao nhanh", không đo "khách SP dễ tính".

Nhưng phân tầng theo số ngày giao chưa chắc là cách đúng. Khách đánh giá theo **lời hứa**: ngày hẹn
giao trung vị ở SP ngắn hơn hẳn, nên 20 ngày là trễ với khách SP mà sớm với khách ở bang xa. Phân
tầng theo trễ / đúng hẹn, SP lại thấp hơn ở cả hai tầng và không có đảo chiều. Ba cách chia, ba câu
chuyện: con số không tự chọn câu chuyện, cơ chế mới chọn.

- Câu hỏi "khách SP có khó tính hơn không": so ở cùng **trải nghiệm so với kỳ vọng** (trễ / đúng
  hẹn), không ở cùng số ngày tuyệt đối.
- Câu hỏi "giao nhanh hơn ở các bang xa có làm review tốt lên không": số ngày giao là thứ ta muốn
  thay đổi, không phân tầng theo nó; nhìn tác động của nó (module 06).

Giới hạn: đây là dữ liệu quan sát; các bang khác SP còn khác về category, seller, giá trị đơn. Tầng
nhanh nhất (0–6 ngày) chênh lệch gần 0, cho thấy "đảo chiều" chủ yếu đến từ các tầng giao chậm, nơi
SP có ít đơn (ví dụ 506 đơn ở nhóm 30+).